In [6]:
import os 
os.chdir("/scale/user/mtannaou/alternance")
from src.visualisation.utils_colormap import CMAP
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import pyresample
import pyproj

cmap_ir = CMAP.cira_ir()
cmap_sar = CMAP.cmap_sar()

def plot_sar(tensor, fig=None, ax=None, cmap=cmap_sar, title=None,
             x_lim=300, x=None, y=None):

    if ax is None:
        ax = plt.gca()

    if x is None:
        y_sar = np.linspace(-x_lim, x_lim, tensor.shape[0])
        x_sar = np.linspace(-x_lim, x_lim, tensor.shape[1])
        x_sar, y_sar = np.meshgrid(x_sar, y_sar)
    else:
        x_sar, y_sar = x, y   # déjà 2D, pas de meshgrid

    im = ax.pcolormesh(
        x_sar,
        y_sar,
        tensor,
        cmap=cmap,
        vmin=0,
        vmax=160/1.94384449,
        shading="auto"
    )

    if x is None:
        ax.set_xlim(-x_lim, x_lim)
        ax.set_ylim(-x_lim, x_lim)

    if title is not None:
        ax.set_title(title)

    if fig is not None:
        fig.colorbar(im, ax=ax, orientation="horizontal")

    ax.set_aspect("equal")


def plot_ir(tensor, fig=None, x=None, y=None, ax=None, x_lim=300,
            cmap=cmap_ir, vmin=-100, vmax=50):

    if ax is None:
        ax = plt.gca()

    tensor = np.squeeze(tensor)
    ny, nx = tensor.shape  # ny = rows, nx = cols

    if x is None:
        x = np.linspace(-x_lim, x_lim, nx)

    if y is None:
        y = np.linspace(-x_lim, x_lim, ny)

    im = ax.pcolormesh(
        x, y, tensor,
        cmap=cmap,
        shading="nearest",
        vmin=vmin,
        vmax=vmax
    )

    if x is None or y is None:
        ax.set_xlim(-x_lim, x_lim)
        ax.set_ylim(-x_lim, x_lim)
    else:
        ax.set_xlim(np.min(x), np.max(x))
        ax.set_ylim(np.min(y), np.max(y))
    ax.set_aspect("equal")

    if fig is not None:
        fig.colorbar(im, ax=ax, orientation="horizontal")


 
def build_storm_centered_dataset(
    ds,
    half_size=500,
    dxy=2.0,
    kind="nearest"
):
    """
    Construit un dataset centré cyclone sur grille régulière [-500, 500] km.

    Returns
    -------
    xr.Dataset
    """

    ir = ds["brightness_temperature"].values
    ir_lon = ((ds["longitude"].values+180)%360)-180
    ir_lat = ds["latitude"].values
    center_lat = ds["storm_latitude"].values[0]
    center_lon = ((ds["storm_longitude"][0].values + 180)%360)-180

    nx = ny = int(2 * half_size / dxy) + 1

    x = np.linspace(-half_size, half_size, nx)
    y = np.linspace(half_size, -half_size, ny)  # nord en haut

    proj = pyproj.Proj(              #place le cyclone au centre (0,0), et je mesure tout en km autou
        proj="aeqd",
        lat_0=center_lat,
        lon_0=center_lon,
        ellps="WGS84",
        units="km"
    )


    longitude, latitude = pyresample.utils.check_and_wrap(ir_lon, ir_lat)
    lon2d, lat2d = np.meshgrid(longitude, latitude)

    swath_ir = pyresample.SwathDefinition(
        lon2d,
        lat2d
    )

    area_def = pyresample.geometry.AreaDefinition(
        "storm",
        "storm centered grid",
        "storm",
        proj.srs,
        nx,
        ny,
        (x[0] - dxy/2, y[-1] - dxy/2, x[-1] + dxy/2, y[0] + dxy/2)
    )

    if kind == "nearest":
        ir_grid = pyresample.kd_tree.resample_nearest(
            swath_ir,
            ir,
            area_def,
            radius_of_influence=100000,
            fill_value=np.nan
        )
    else:
        ir_grid = pyresample.kd_tree.resample_gauss(
            swath_ir,
            ir,
            area_def,
            radius_of_influence=100000,
            sigmas=25000
        )

   

    ds["ir_aeqd"] = xr.DataArray(
            ir_grid,
            dims=("y", "x"),
            coords={
                "x": x,
                "y": y,
            },
            attrs={
                "units": ds["brightness_temperature"].attrs.get("units", ""),
                "description": "Brightness temperature projected onto storm-centered AEQD grid",
            },
        )

    ds["center_lat"] = center_lat
    ds["center_lon"] = center_lon

    return ds

In [7]:
import os
os.chdir("/scale/user/mtannaou/alternance/src/IR_to_SAR/ML_IR_SAR")
os.getcwd()

'/scale/user/mtannaou/alternance/src/IR_to_SAR/ML_IR_SAR'

In [22]:
for itr, row in tqdm(dataset_valide.iterrows(), total=len(dataset_valide), desc=f"Generating data Using IRAR Dataset : ..............."):
    if not row["valide_sargeo"]:
        continue

    cyclone_id = row["cyclone_id"]
    year = row["year"]
    path_irar = row["path_irar"]
    path_l2 = row["L2M path"]
    date_irar = datetime.strptime(os.path.basename(path_irar).split("_s")[-1].split("_")[0], "%Y%m%d%H%M%S")

    sequence_path_plus =  []
    sequence_path_moins = []

    for i in range(1,5):
        date_i = date_irar + timedelta(minutes=i*30)
        date_j = date_irar - timedelta(minutes=i*30)
        
        sub_i = irar[(irar["date_irar"]==date_i) & (irar["cyclone_id"] == cyclone_id)]
        sub_j = irar[(irar["date_irar"]==date_j) & (irar["cyclone_id"] == cyclone_id)]

        sequence_path_plus.append(sub_i["path_nc"].values[0])
        sequence_path_moins.append(sub_j["path_nc"].values[0])
    
    train_seq_paths = sequence_path_moins + [path_irar] + sequence_path_plus

    for j, path in enumerate(train_seq_paths):
        if "aeqd" in path:
            continue  # Skip already processed files
        try:
            ds = xr.open_dataset(path)
            ds = ds.where(ds != -999.0)

            ds = build_storm_centered_dataset(ds)

            tmp_path = path + ".tmp"

            ds.to_netcdf(tmp_path)

            os.replace(tmp_path, path)

            dataset.loc[itr, "ir_regrided_sargeo"] = True


        except Exception as e:
            print(f"Error loading {path}: {e}")
            dataset.loc[itr, "ir_regrided_sargeo"] = False
            break
    
            

Generating data Using IRAR Dataset : ...............:   3%|▎         | 33/1160 [01:54<53:18,  2.84s/it]  

Error loading /scale/project/ifremer-isi-jumeaunumerique/IRAR_BestTrack_corrected/2024/al182024/TC-IRAR_v02r02_AL182024_s20241110122617_e20241110122617.nc: Invalid projection: +proj=aeqd +lat_0=nan +lon_0=nan +ellps=WGS84 +units=km +type=crs: (Internal Proj Error: proj_create: invalid value for lat_0)


Generating data Using IRAR Dataset : ...............:   7%|▋         | 83/1160 [04:34<52:34,  2.93s/it]  

Error loading /scale/project/ifremer-isi-jumeaunumerique/IRAR_BestTrack_corrected/2024/al162024/TC-IRAR_v02r02_AL162024_s20241022120117_e20241022120117.nc: [Errno 2] No such file or directory: '/scale/project/ifremer-isi-jumeaunumerique/IRAR_BestTrack_corrected/2024/al162024/TC-IRAR_v02r02_AL162024_s20241022120117_e20241022120117.nc'


Generating data Using IRAR Dataset : ...............:   8%|▊         | 87/1160 [04:44<43:15,  2.42s/it]

...

Generating data Using IRAR Dataset : ...............:  45%|████▍     | 519/1160 [29:52<35:19,  3.31s/it]

Error loading /scale/project/ifremer-isi-jumeaunumerique/IRAR_BestTrack_corrected/2023/al042023/TC-IRAR_v02r02_AL042023_s20230626000020_e20230626000020.nc: Invalid projection: +proj=aeqd +lat_0=nan +lon_0=nan +ellps=WGS84 +units=km +type=crs: (Internal Proj Error: proj_create: invalid value for lat_0)


Generating data Using IRAR Dataset : ...............:  69%|██████▉   | 801/1160 [46:15<18:50,  3.15s/it]

Error loading /scale/project/ifremer-isi-jumeaunumerique/IRAR_BestTrack_corrected/2021/al042021/TC-IRAR_v02r02_AL042021_s20210629002117_e20210629002117.nc: Invalid projection: +proj=aeqd +lat_0=nan +lon_0=nan +ellps=WGS84 +units=km +type=crs: (Internal Proj Error: proj_create: invalid value for lat_0)


Generating data Using IRAR Dataset : ...............: 100%|██████████| 1160/1160 [1:06:55<00:00,  3.46s/it]


## regrillage de tous les infra rouges de IRAR

In [ ]:
irar_path = "/scale/project/ifremer-isi-jumeaunumerique/IRAR_BestTrack_corrected"
import os; import xarray as xr;from tqdm import tqdm

totale = 0
valide = 0
for year in tqdm(os.listdir(irar_path), desc="Regrided All IRAR samples :", total=len(os.listdir(irar_path))):
    year_path = os.path.join(irar_path,year)
    for cyc_id in os.listdir(year_path):
        cyc_id_path = os.path.join(year_path,cyc_id)
        for path in os.listdir(cyc_id_path):
            if "aeqd" in path :
                continue
            totale += 1
            path_nc = os.path.join(cyc_id_path,path)

            try : 
                ds = xr.open_dataset(path_nc)
                ds = ds.where(ds != -999.0)
                ds = build_storm_centered_dataset(ds)

                tmp_path = path_nc + ".tmp"

                ds.to_netcdf(tmp_path)

                os.replace(tmp_path, path_nc)
                valide += 1
            except  Exception as e:
                print("error",e)
                continue
                
                


Regrided All IRAR samples ::   0%|          | 0/9 [00:16<?, ?it/s]


KeyboardInterrupt: 